In [10]:
from unittest import result

#加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessageChunk

load_dotenv()

True

In [11]:
#定义模型
from langchain.chat_models import init_chat_model
import os

model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

In [12]:
#定义工具
from langchain_tavily import TavilySearch
web_search=TavilySearch(
    max_results=5,
    topic="general"
)

In [13]:
#添加记忆管理
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
#连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
#初始化checkpointer
checkpointer = SqliteSaver(connection)
#自动建表
checkpointer.setup()

In [14]:
from langchain.agents import create_agent

system_prompt="""
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作:
1.识别和评估食材:若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索:优先调用web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序:从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4，结构化方案输出:把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。
请严格按照流程，优先调web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)


In [15]:
from langchain.messages import HumanMessage
multimodal_message=HumanMessage([
    {"type": "text","text":"帮我看看能做什么."},
    {"type": "image","url":"https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"},
])
config={"configurable":{"thread_id":"2"}}

In [29]:
response = agent.invoke({"messages":[multimodal_message]},config)

In [31]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_364886c5787845bbb924535c)
 Call ID: call_364886c5787845bbb924535c
  Args:
    query: 西红柿 青椒 豆腐 西兰花 茄子 家常菜谱 简单营养
    search_depth: advanced
    include_images: True
================================= Tool Message =================================
Name: tavily_search

{"query": "西红柿 青椒 豆腐 西兰花 茄子 家常菜谱 简单营养", "follow_up_questions": null, "answer": null, "images": ["https://lookaside.fbsbx.com/lookaside/crawler/media/?media_id=10242614606170993", "https://www.tiktok.com/api/img/?itemId=7218526553855282478&location=0&aid=1988", "https://i.ytimg.com/vi/6gl9vcIcs-c/sddefault.jpg", "https://s4.cdn.jiaonizuocai.com/caipu/201609/1722/172216151248.jpg/NjAwX2MyX

In [32]:
response=agent.invoke(
    {"messages":[HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [33]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_364886c5787845bbb924535c)
 Call ID: call_364886c5787845bbb924535c
  Args:
    query: 西红柿 青椒 豆腐 西兰花 茄子 家常菜谱 简单营养
    search_depth: advanced
    include_images: True
================================= Tool Message =================================
Name: tavily_search

{"query": "西红柿 青椒 豆腐 西兰花 茄子 家常菜谱 简单营养", "follow_up_questions": null, "answer": null, "images": ["https://lookaside.fbsbx.com/lookaside/crawler/media/?media_id=10242614606170993", "https://www.tiktok.com/api/img/?itemId=7218526553855282478&location=0&aid=1988", "https://i.ytimg.com/vi/6gl9vcIcs-c/sddefault.jpg", "https://s4.cdn.jiaonizuocai.com/caipu/201609/1722/172216151248.jpg/NjAwX2MyX

In [18]:
from langchain.messages import AIMessageChunk
for chunk,metadata in agent.stream(
    {"messages":[multimodal_message]},
    config,
    stream_mode="messages"
):
    if isinstance(chunk,AIMessageChunk) and chunk.content:
        print(chunk.content,end="",flush=True)

您好！您的冰箱食材非常丰富新鲜，我注意到这是之前分享过的同一批食材。让我为您快速总结一下**最佳使用方案**：

---

## 🎯 3道今日推荐菜

| 推荐 | 菜名 | 用时 | 核心食材 |
|------|------|------|----------|
| 🥇 | 彩椒西兰花炒虾仁 | 15分钟 | 虾仁+西兰花+青椒 |
| 🥈 | 西红柿豆腐烩虾仁 | 20分钟 | 豆腐+西红柿+虾仁 |
| 🥉 | 茄子烧豆角 | 25分钟 | 茄子+豆角 |

---

## ⏰ 食材消耗优先级

```
🔴 2-3天内用完：虾仁、绿叶菜、西兰花
🟡 3-5天内用完：豆腐、西红柿、豆角
🟢 5-7天内用完：茄子、青椒、白菜、腊肠酱
```

---

## ❓ 您需要什么帮助？

请告诉我您想了解：

1. **某道菜的详细做法**（比如彩椒西兰花炒虾仁的完整步骤）
2. **口味调整**（更辣/更清淡/少油等）
3. **特定食材的做法**（比如豆腐怎么做最好吃）
4. **一周完整菜谱规划**
5. **其他需求**

请告诉我您的需求，我来为您详细解答！😊

In [19]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-05-11T13:33:44.161206+00:00',
 'id': '1f14d3e0-7ef0-660c-8021-832f0e82c327',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}], additional_kwargs={}, response_metadata={}, id='80d7717a-472f-43a5-85f0-484e73da517a'),
   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 4491, 'total_tokens': 4693, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 3, 'rejected_prediction_tokens': None, 'text_tokens': 199}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 2496, 'text_tokens': 1995}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-cb6d4129-581a-9b80-856f-e7405410bb64', 'finish_reason': '

In [20]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}] additional_kwargs={} response_metadata={} id='80d7717a-472f-43a5-85f0-484e73da517a'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 4491, 'total_tokens': 4693, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 3, 'rejected_prediction_tokens': None, 'text_tokens': 199}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 2496, 'text_tokens': 1995}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-cb6d4129-581a-9b80-856f-e7405410bb64', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e0cbd-2365-7fc2-b73

In [22]:
from langchain.messages import AIMessage,HumanMessage

def get_messages(thread_id: str) -> list[dict[str,str]]:
    """获取会话历史"""
    cp=checkpointer.get({"configurable":{"thread_id":thread_id}})
    #如果不存在，返回空列表
    if not cp:
        return []
    #安全获取messages
    channel_values=cp.get("channel_values")
    if not channel_values:
        return []

    message=channel_values.get("messages",[])
    if not message:
        return []

    #转换消息格式
    result=[]
    for msg in message:
        if not msg.content:
            continue

        if isinstance(msg,HumanMessage):
            result.append({"role":"user","content":msg.content})
        elif isinstance(msg,AIMessage):
            result.append({"role":"assistant","content":msg.content})

    return result

In [25]:
get_messages("2")

[]

In [24]:
checkpointer.delete_thread("2")